# Explicit redocking evaluation

Public `dockingmt.evaluate_redocking` produces a detached JSON report in angstrom.
The caller chooses the cutoff; top-N follows current result positions. No ligand
alignment, symmetry correction or chemistry validation is performed.

These three seeded native Vina controls use provisional 181L preparation and
externally prepared 1IEP, including a displaced search-domain control. 1IEP uses
all 40 retained PDBQT atoms, including polar hydrogens, against the pinned prepared
input conformation. It is different from the historical 37-heavy-atom SDF metric.
No dataset success rate or biological affinity conclusion is inferred.
See [the evaluation contract](redocking_evaluation.md).


In [1]:
from pathlib import Path
import json
import platform
import importlib.metadata as metadata
import molsysmt as msm
import dockingmt as dmt

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'tests/data/vina_torsions/1iep_ligand.pdbqt').is_file())
import sys
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from devtools.qualify_redocking_evaluation import qualify, summary
print(json.dumps({
    'python': platform.python_version(), 'vina': metadata.version('vina'),
    'molsysmt_origin': msm.__file__, 'dockingmt_origin': dmt.__file__,
}, indent=2))


{
  "python": "3.14.7",
  "vina": "1.2.7",
  "molsysmt_origin": "/home/diego/repos@uibcdf/dockingmt/.pytest_cache/codex-source-archives/dockingmt-template-provider-c19a47ada0c2279029abfa296cf915560610ad9a/molsysmt/__init__.py",
  "dockingmt_origin": "/home/diego/repos@uibcdf/dockingmt/dockingmt/__init__.py"
}


In [2]:
reports = qualify()
assert reports['181l_provisional']['first_pose']['recovered']
assert reports['1iep_external']['first_pose']['recovered']
assert not any(row['recovered'] for row in reports['1iep_displaced_domain']['poses'])
print(json.dumps(summary(reports), indent=2))


{
  "181l_provisional": {
    "n_poses": 1,
    "rmsds_angstrom": [
      2.2916067143673082
    ],
    "top_n": [
      {
        "requested": 1,
        "considered": 1,
        "recovered": true
      },
      {
        "requested": 5,
        "considered": 1,
        "recovered": true
      }
    ],
    "atom_correspondence": "verified_source_atom_keys",
    "preparation": "provisional"
  },
  "1iep_external": {
    "n_poses": 1,
    "rmsds_angstrom": [
      0.8760362292736524
    ],
    "top_n": [
      {
        "requested": 1,
        "considered": 1,
        "recovered": true
      },
      {
        "requested": 5,
        "considered": 1,
        "recovered": true
      }
    ],
    "atom_correspondence": "caller_declared_positional",
    "preparation": "unassessed"
  },
  "1iep_displaced_domain": {
    "n_poses": 3,
    "rmsds_angstrom": [
      25.247243153461323,
      22.874961337672243,
      22.243653168937875
    ],
    "top_n": [
      {
        "requested": 1,
     

In [3]:
saved = json.dumps(reports, allow_nan=False, indent=2)
restored = json.loads(saved)
assert restored == reports
assert restored['181l_provisional']['context']['preparation']['partner']['assessment'] == 'provisional'
assert restored['1iep_external']['context']['preparation']['partner']['assessment'] == 'unassessed'
assert all(report['criterion']['alignment'] == 'none'
           and report['criterion']['symmetry_correction'] == 'none'
           and report['criterion']['unit'] == 'angstrom'
           for report in restored.values())
output = root / 'devguide/validation/data/redocking_evaluation/cases.json'
output.write_text(saved + '\n')
print({'saved_reports': len(restored), 'path': str(output.relative_to(root)),
       'interpretation': 'single-case software controls, no dataset success rate'})


{'saved_reports': 3, 'path': 'devguide/validation/data/redocking_evaluation/cases.json', 'interpretation': 'single-case software controls, no dataset success rate'}
